# European Option Pricing

Price an at-the-money European call using three methods: closed-form Black-Scholes, Monte Carlo, and the Cox-Ross-Rubinstein binomial tree.

In [1]:
from options_pricer import AmericanOption, EuropeanOption, OptionType, MarketData, BlackScholesModel
from options_pricer.pricers import ClosedFormPricer, FiniteDifferencesPricer, MonteCarloPricer, TreePricer

## Setup

In [2]:
option = EuropeanOption(option_type=OptionType.CALL, strike=100.0, expiry=1.0)
market = MarketData(spot=100.0, rate=0.05, div_yield=0.02)
model  = BlackScholesModel(vol=0.20)

## Closed-form (Black-Scholes)

In [3]:
cf_price  = ClosedFormPricer.price(option, market, model)
cf_greeks = ClosedFormPricer.greeks(option, market, model)

print(f"Price : {cf_price:.4f}")
print(f"Delta : {cf_greeks.delta:.4f}")
print(f"Gamma : {cf_greeks.gamma:.4f}")
print(f"Vega  : {cf_greeks.vega:.4f}")
print(f"Theta : {cf_greeks.theta:.4f}")
print(f"Rho   : {cf_greeks.rho:.4f}")

Price : 9.2270
Delta : 0.5869
Gamma : 0.0190
Vega  : 0.3790
Theta : -0.0139
Rho   : 0.4946


## Monte Carlo

In [4]:
mc_price = MonteCarloPricer(n_paths=1_000_000, seed=42).price(option, market, model)
print(f"Price : {mc_price:.4f}  (error vs BS: {mc_price - cf_price:+.4f})")

Price : 9.2297  (error vs BS: +0.0027)


## Binomial Tree (CRR)

In [5]:
tree_price = TreePricer(n_steps=1000).price(option, market, model)
print(f"Price : {tree_price:.4f}  (error vs BS: {tree_price - cf_price:+.4f})")

Price : 9.2251  (error vs BS: -0.0019)


## Finite Differences (Crank-Nicolson)

In [6]:
fd_price = FiniteDifferencesPricer(n_s=400, n_t=400).price(option, market, model)
print(f"Price : {fd_price:.4f}  (error vs BS: {fd_price - cf_price:+.4f})")

Price : 9.2280  (error vs BS: +0.0010)


## Comparison

In [7]:
import pandas as pd

pd.DataFrame({
    "Method":      ["Closed-form", "Monte Carlo", "Binomial Tree", "Finite Differences"],
    "Price":       [cf_price, mc_price, tree_price, fd_price],
    "Error vs BS": [0.0, mc_price - cf_price, tree_price - cf_price, fd_price - cf_price],
}).set_index("Method").round(4)

,Price,Error vs BS
Method,,
Closed-form,9.2270,0.0000
Monte Carlo,9.2297,0.0027
Binomial Tree,9.2251,-0.0019
Finite Differences,9.2280,0.0010


# American Option Pricing

Price an at-the-money American put using three methods: binomial tree, Longstaff-Schwartz Monte Carlo, and Crank-Nicolson finite differences. The comparison table also shows the early exercise premium over the equivalent European put.

## Setup

An at-the-money American put. The put has meaningful early exercise value — unlike a call on a dividend-paying stock, a put can be worth more exercised immediately when deep in-the-money.

In [8]:
am_option = AmericanOption(option_type=OptionType.PUT, strike=100.0, expiry=1.0)

## Binomial Tree (CRR)

In [9]:
am_tree_price = TreePricer(n_steps=1000).price(am_option, market, model)
print(f"Price : {am_tree_price:.4f}")

Price : 6.6598


## Monte Carlo (Longstaff-Schwartz)

In [10]:
am_mc_price = MonteCarloPricer(n_paths=500_000, n_steps=100, seed=42).price(am_option, market, model)
print(f"Price : {am_mc_price:.4f}  (error vs Tree: {am_mc_price - am_tree_price:+.4f})")

Price : 6.6201  (error vs Tree: -0.0397)


## Finite Differences (Crank-Nicolson)

In [11]:
am_fd_price = FiniteDifferencesPricer(n_s=400, n_t=400).price(am_option, market, model)
print(f"Price : {am_fd_price:.4f}  (error vs Tree: {am_fd_price - am_tree_price:+.4f})")

Price : 6.6604  (error vs Tree: +0.0006)


## Comparison

In [12]:
import pandas as pd

eu_put = EuropeanOption(option_type=OptionType.PUT, strike=100.0, expiry=1.0)
eu_put_price = ClosedFormPricer.price(eu_put, market, model)

pd.DataFrame({
    "Method":                 ["Binomial Tree", "Monte Carlo (LSM)", "Finite Differences"],
    "Price":                  [am_tree_price, am_mc_price, am_fd_price],
    "Error vs Tree":          [0.0, am_mc_price - am_tree_price, am_fd_price - am_tree_price],
    "Early Exercise Premium": [am_tree_price - eu_put_price, am_mc_price - eu_put_price, am_fd_price - eu_put_price],
}).set_index("Method").round(4)

,Price,Error vs Tree,Early Exercise Premium
Method,,,
Binomial Tree,6.6598,0.0000,0.3297
Monte Carlo (LSM),6.6201,-0.0397,0.2900
Finite Differences,6.6604,0.0006,0.3303
